# Qwen3-VL-8B-Instruct — Large VLM Baseline

> **GPU required:** A100 (40 GB). **No smaller GPU will work at fp16.**
> In Colab: *Runtime → Change runtime type → A100.*
> This model needs ~28 GB VRAM. L4 (24 GB) only works with 4-bit quantization (see cell 4).

Transcribe evaluation subset pages using
[Qwen3-VL-8B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct),
an 8.29-billion-parameter vision-language model from Alibaba's Qwen team.

### What makes Qwen3-VL interesting

- **Architecture**: Built on the **Qwen3-VL** vision-language framework, which
  pairs a NaViT-style vision encoder with the Qwen3 language backbone. Unlike
  fixed-resolution ViTs, NaViT processes images at **native aspect ratio** using
  dynamic resolution — the image is tiled into patches whose count scales with
  pixel budget, not a fixed grid
  ([paper](https://huggingface.co/papers/2502.13923)).
- **OCR capability**: Qwen3-VL was trained on large-scale OCR and document
  understanding data, making it one of the strongest open-weight models for
  text-in-image tasks — including handwritten text, though it was not
  specifically fine-tuned for historical manuscripts.
- **Design goal**: General-purpose multimodal assistant. By evaluating it on
  our HTR benchmark we measure how well a large generalist VLM performs
  compared to specialist models like CHURRO and TRIDIS.

| Detail | Value |
|--------|-------|
| HuggingFace ID | [`Qwen/Qwen3-VL-8B-Instruct`](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct) |
| Parameters | 8.29 B |
| Base architecture | Qwen3-VL (NaViT + Qwen3 LM) |
| VRAM required | ~28 GB (needs A100 40 GB) |
| dtype | `float16` |
| Context length | 32K tokens |
| OCR languages | Multilingual (Chinese, English, Spanish, etc.) |

### Relationship to CHURRO

CHURRO-3B (notebook 01) is **fine-tuned from Qwen2.5-VL-3B-Instruct** — the
3B sibling of this model's predecessor (Qwen2.5-VL). Comparing their results
reveals whether HTR-specific fine-tuning on 99K manuscript pages (CHURRO)
outperforms a larger general-purpose VLM (Qwen3-VL-8B). This is one of the
most interesting comparisons in our benchmark.

### Chat template for VLMs

Modern VLMs use a **chat template** format for input, where images and text
are structured as conversation messages. The `apply_chat_template()` method
handles formatting automatically — converting a list of `messages` dicts
into the model's expected token sequence with proper special tokens.

**Note on model variants:** Qwen3-VL comes in two instruction-tuned variants:
`-Instruct` (standard chat) and `-Thinking` (extended reasoning with
`enable_thinking`). We use the `-Instruct` variant, which has **no thinking
mode logic** in its chat template — only the `-Thinking` variant supports it.

## 1. Setup & Data Access

We mount Google Drive so the notebook can read the evaluation images that have
been pre-uploaded to `MyDrive/paleo-ocr/subset_images/`. Four path variables
control where inputs are read from and where outputs are written:

- **`DRIVE_BASE`** — root of the paleo-ocr project on Drive.
- **`IMAGES_DIR`** — folder containing the manuscript page images.
- **`MANIFEST_PATH`** — JSON file listing every page to transcribe (page ID,
  canonical file name, document metadata).
- **`OUTPUT_DIR`** — where raw transcription `.txt` files are saved (one per page).

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Cache HuggingFace models on local storage (wiped when session ends, saves Drive space)
os.environ['HF_HOME'] = '/content/.hf_cache'

# Authenticate with HuggingFace for faster downloads (optional)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except (ImportError, userdata.SecretNotFoundError):
    print('HF_TOKEN not found — downloads will be unauthenticated (slower)')

DRIVE_BASE = '/content/drive/MyDrive/paleo-ocr'
IMAGES_DIR = f'{DRIVE_BASE}/subset_images'
MANIFEST_PATH = f'{DRIVE_BASE}/evaluation_subset.json'
OUTPUT_DIR = f'{DRIVE_BASE}/data/results/qwen3_vl_8b/raw'

## 2. Install Dependencies

| Package | Purpose |
|---------|--------|
| `transformers` | Model and processor classes (`Qwen3VLForConditionalGeneration`, `AutoProcessor`) |
| `torch` | Tensor operations, GPU inference |
| `accelerate` | `device_map='auto'` support for automatic GPU placement |
| `bitsandbytes` | Only needed if using the 4-bit quantization fallback for L4 GPUs |
| `pillow` | Image loading and conversion |
| `qwen-vl-utils` | Qwen's vision-language utilities for image processing; required by `AutoProcessor` for Qwen VL models |

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes pillow qwen-vl-utils

## 3. Load Manifest & Prepare Output

The manifest is a JSON list where each entry contains at minimum:

```json
{"page_id": "doc001_p01", "canonical_name": "doc001_p01.jpg", ...}
```

We iterate over every entry later. The output directory is created if it does
not already exist. **Resumability**: any page whose output file already exists
and is non-empty will be skipped during transcription, so the notebook can be
re-run after interruptions without re-processing finished pages.

In [ ]:
import json, os
from pathlib import Path
from PIL import Image
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
print(f'Loaded {len(manifest)} pages')

# >>> TRIAL RUN: process only CODEA-0143 (2 pages) — remove this line for full evaluation
manifest = [e for e in manifest if e['doc_id'] == 'CODEA-0143']
print(f'  Filtered to {len(manifest)} pages (trial run: CODEA-0143 only)')

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Load Model

Qwen3-VL-8B-Instruct is loaded with **`Qwen3VLForConditionalGeneration`** —
the dedicated model class for Qwen3-VL architecture in transformers ≥ 4.57.
This replaces the older `Qwen2_5_VLForConditionalGeneration` class, which does
not correctly load Qwen3-VL weights
([model card](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct)).

Key options:

- **`dtype=torch.float16`** — full half-precision for maximum transcription
  quality. The `dtype` kwarg is the canonical form in transformers 5.x
  (replacing the older `torch_dtype`).
- **`device_map='auto'`** — lets `accelerate` place layers on the GPU
  automatically.
- **`trust_remote_code=True`** — required because the model repo ships custom
  processor code (`Qwen3VLProcessor`).

The attention backend is **auto-detected** by transformers — it will use Flash
Attention 2 on Ampere+ GPUs (A100), SDPA on Turing (T4/L4), or eager as
fallback. No need to hardcode `attn_implementation`.

### Memory budget (fp16)

| Component | VRAM |
|-----------|------|
| Model weights | ~16.6 GB |
| Image attention + KV cache | ~12 GB |
| **Total** | **~28 GB** |

This fits on an A100 (40 GB) with ~12 GB headroom.

### 4-bit fallback (L4 24 GB)

If fp16 causes OOM on your GPU (e.g. L4 24 GB), uncomment the
`BitsAndBytesConfig` block below to reduce model weights to ~5 GB via NF4
quantization. This trades some transcription quality for fitting in less VRAM.

### Image resolution (`max_pixels`)

Qwen3-VL ships with `max_pixels = 16,777,216` (~16.8 MP) — essentially
uncapped. While this means no downsampling for any of our images, the largest
CODEA pages (~12 MP) would generate so many vision tokens that they can OOM
even on an A100.

We cap `max_pixels` at **5,017,600** (~5 MP), which provides:

- **Toledo** images (1.8–2.6 MP) at full resolution — no downsampling.
- **CODEA** images (8–12 MP) at ~2.4× downsampling (better than the 3× cap
  on 3B models like CHURRO).

The transcription loop also includes an **OOM fallback**: if a large image
still exceeds VRAM, it retries at half the `max_pixels` budget rather than
crashing.

In [ ]:
MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'

# --- Uncomment below if fp16 causes OOM on your GPU (e.g. L4 24 GB) ---
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type='nf4',
#     bnb_4bit_compute_dtype=torch.float16,
# )

processor = AutoProcessor.from_pretrained(
    MODEL_ID, trust_remote_code=True, max_pixels=5_017_600
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
    # quantization_config=bnb_config,  # Uncomment if using 4-bit fallback above
)
print('Model loaded (fp16)')
print(f'max_pixels: {processor.image_processor.max_pixels}')

## 5. Transcribe Pages

**Prompt strategy:** We use a detailed prompt asking the model to preserve
original spelling, expand abbreviations, and maintain line breaks. This is the
most detailed prompt in our benchmark — larger models can follow complex
instructions better than smaller ones.

### Unified inference pattern

We use the **unified `processor.apply_chat_template()`** call, which handles
chat formatting, image encoding, and tokenization in a single step:

```python
inputs = processor.apply_chat_template(
    messages, tokenize=True, return_dict=True, return_tensors='pt'
)
```

This replaces the older two-step pattern (`apply_chat_template(tokenize=False)`
→ `processor(text=..., images=...)`) and is the recommended approach for
Qwen3-VL. The PIL `Image` object is passed directly in the `messages` dict
via the `"image"` key.

### Generation parameters

| Parameter | Value | Rationale | Source |
|-----------|-------|-----------|--------|
| `max_new_tokens` | `2048` | Sufficient for full manuscript page transcription | Project convention |
| `do_sample` | `True` | Required for temperature > 0 | — |
| `temperature` | `0.7` | Official recommended VL setting | [model card](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct) |
| `top_p` | `0.8` | Nucleus sampling cutoff | [model card](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct) |
| `top_k` | `20` | Limits token candidate pool | [model card](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct) |

**Note:** The model card also recommends `repetition_penalty=1.05` for vLLM
deployments, but this is a vLLM-specific parameter and is not available in
HuggingFace `generate()`. For greedy decoding (deterministic output), set
`do_sample=False` and remove the temperature/top_p/top_k parameters.

### Output trimming

The official Qwen3-VL pattern trims input tokens from the output using a list
comprehension:

```python
[out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs['input_ids'], output_ids)]
```

This is more robust than the simpler `output_ids[:, inputs.input_ids.shape[1]:]`
slice, as it handles variable-length inputs correctly in batched scenarios.

### Context manager

`torch.inference_mode()` is used instead of `torch.no_grad()` for slightly
better performance — it disables additional autograd tracking beyond what
`no_grad()` covers
([PyTorch docs](https://pytorch.org/docs/stable/generated/torch.inference_mode.html)).

In [ ]:
PROMPT = 'Transcribe the handwritten text in this manuscript image exactly as written. Preserve original spelling. Expand abbreviations. Maintain line breaks. If illegible, write [...].'

MAX_PIXELS = processor.image_processor.max_pixels

for i, entry in enumerate(manifest):
    page_id = entry['page_id']
    out_path = Path(OUTPUT_DIR) / f'{page_id}_raw.txt'
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f'[{i+1}/{len(manifest)}] {page_id} - SKIP')
        continue

    img_path = Path(IMAGES_DIR) / entry['canonical_name']
    if not img_path.exists():
        print(f'[{i+1}] {page_id} - NOT FOUND')
        continue

    try:
        image = Image.open(img_path).convert('RGB')
        w, h = image.size
        print(f'  Image: {w}x{h} ({w*h/1e6:.1f} MP)')

        messages = [
            {'role': 'user', 'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': PROMPT},
            ]}
        ]

        try:
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                return_dict=True,
                return_tensors='pt',
                add_generation_prompt=True,
            )
            inputs.pop('token_type_ids', None)
            inputs = {k: v.to(model.device) for k, v in inputs.items() if hasattr(v, 'to')}

            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=2048,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.8,
                    top_k=20,
                )

        except torch.cuda.OutOfMemoryError:
            # OOM fallback: clear cache and retry at half resolution
            torch.cuda.empty_cache()
            fallback_pixels = MAX_PIXELS // 2
            print(f'  OOM at max_pixels={MAX_PIXELS}, retrying with {fallback_pixels}')

            # Temporarily reduce processor resolution for retry
            processor.image_processor.max_pixels = fallback_pixels
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                return_dict=True,
                return_tensors='pt',
                add_generation_prompt=True,
            )
            processor.image_processor.max_pixels = MAX_PIXELS  # Restore
            inputs.pop('token_type_ids', None)
            inputs = {k: v.to(model.device) for k, v in inputs.items() if hasattr(v, 'to')}

            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=2048,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.8,
                    top_k=20,
                )

        # Trim input tokens from output (official Qwen3-VL pattern)
        generated = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs['input_ids'], output_ids)]
        text = processor.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        out_path.write_text(text, encoding='utf-8')
        print(f'[{i+1}/{len(manifest)}] {page_id} - {len(text)} chars')

    except Exception as e:
        print(f'[{i+1}] {page_id} - ERROR: {e}')

    # Free cached GPU memory between pages to prevent fragmentation
    torch.cuda.empty_cache()

print('Done!')

## 6. Export Results

All raw transcription files are compressed into a single ZIP archive and
offered for download via `google.colab.files`. The archive mirrors the flat
structure of `OUTPUT_DIR` (`{page_id}_raw.txt` per page), which is the format
expected by the downstream CER/WER evaluation scripts.

After downloading, the ZIP can be fed directly into the evaluation pipeline
(`data/evaluation/`) which computes character error rate (CER), word error rate
(WER), and additional semantic and statistical metrics against the ground
truth transcriptions.

In [ ]:
import shutil
shutil.make_archive('/content/qwen3_vl_8b_results', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/qwen3_vl_8b_results.zip')